### Create a Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [0]:
cust_info_schema = StructType([
    StructField('cst_id', IntegerType(), True),
    StructField('cst_key', StringType(), True),
    StructField('cst_firstname', StringType(), True),
    StructField('cst_lastname', StringType(), True),
    StructField('cst_marital_status', StringType(), True),
    StructField('cst_gndr', StringType(), True),
    StructField('cst_create_date', DateType(), True)
])

In [0]:
prd_info_schema = StructType([
    StructField('prd_id', IntegerType(), True),
    StructField('prd_key', StringType(), True),
    StructField('prd_nm', StringType(), True),
    StructField('prd_cost', DoubleType(), True),
    StructField('prd_line', StringType(), True),
    StructField('prd_start_dt', DateType(), True),
    StructField('prd_end_dt', DateType(), True)
])

In [0]:
sales_details_schema = StructType([
    StructField('sls_ord_num', StringType(), True),
    StructField('sls_prd_key', StringType(), True),
    StructField('sls_cst_id', IntegerType(),True),
    StructField('sls_order_dt', IntegerType(), True),
    StructField('sls_ship_dt', IntegerType(), True),
    StructField('sls_due_dt', IntegerType(), True),
    StructField('sls_sales', DoubleType(), True),
    StructField('sls_quantity', IntegerType(), True),
    StructField('sls_price', DoubleType(), True)
])

In [0]:
cust_az12_schema = StructType([
    StructField('cid', StringType(), True),
    StructField('bdate', DateType(), True),
    StructField('gen', StringType(), True)
])

In [0]:
loc_a101_schema = StructType([
    StructField('cid', StringType(), True),
    StructField('cntry', StringType(), True)
])

In [0]:
px_cat_g1v2_schema = StructType([
    StructField('id', StringType(), True),
    StructField('cat', StringType(), True),
    StructField('subcat', StringType(), True),
    StructField('maintenance', StringType(), True)
])

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS test01.bronze;")

DataFrame[]

### Load Data and Create Table

In [0]:
sources = [
    ('crm/cust_info.csv', cust_info_schema, 'crm_cust_info'),
    ('crm/prd_info.csv', prd_info_schema, 'crm_prd_info'),
    ('crm/sales_details.csv', sales_details_schema, 'crm_sales_details'),
    ('erp/CUST_AZ12.csv', cust_az12_schema, 'erp_cust_az12'),
    ('erp/LOC_A101.csv', loc_a101_schema, 'erp_loc_a101'),
    ('erp/PX_CAT_G1V2.csv', px_cat_g1v2_schema, 'erp_px_cat_g1v2')
]

base = '/Volumes/test01/bronze/source/'

In [0]:
for path, schema, table in sources:
    (
        spark.read.format('csv')
        .schema(schema)
        .option('header', True)
        .load(base + path)
        .withColumn('_ingested_at', current_timestamp())
        .withColumn('_source_file', col('_metadata.file_path'))
        .write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', True)
        .saveAsTable(f'test01.bronze.{table}')
    )
    print(f'{table}: {spark.table(f'test01.bronze.{table}').count()} rows')

crm_cust_info: 18494 rows
crm_prd_info: 397 rows
crm_sales_details: 60398 rows
erp_cust_az12: 18484 rows
erp_loc_a101: 18484 rows
erp_px_cat_g1v2: 37 rows
